<a href="https://colab.research.google.com/github/hmatthe8/Data-mining/blob/main/LAB/lab_04_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4 — Exercise: Exploring World Stock Market Indices

**ITCS 3162 — Introduction to Data Mining**

**Name:** _your name here_
**Date:** _today's date_

You'll explore daily prices for six major world stock indices over five years (2020–2024). Unlike penguins, this is **time-series data with a categorical dimension (ticker)** — you'll need to think about both the temporal structure and cross-sectional comparisons.

You have **two options** for loading the data — use whichever you prefer:

- **Option A (default):** Load the included `world_indices.csv` — pre-built, works offline, gives consistent results.
- **Option B (live fetch):** Download fresh data directly from Yahoo Finance using the `yfinance` library. This is how real analysts pull market data, but it requires internet and Yahoo's API can be flaky.

Both options produce a DataFrame with the same columns, so the rest of the notebook works identically.

| Column | Notes |
|---|---|
| `Date` | Trading day |
| `Ticker` | Yahoo Finance symbol (e.g. `^GSPC` for S&P 500) |
| `Name` | Full index name |
| `Country` | Country |
| `Open`, `High`, `Low`, `Close` | Daily prices |
| `Volume` | Trading volume (has some missing values) |

When you're done, **Restart & Run All**, download as `.ipynb`, and submit via Canvas.


## Setup — imports


In [50]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

## Load the data — pick ONE option

Set `USE_LIVE_FETCH = False` for the default CSV (recommended), or `True` to try the live yfinance API. Then run the cell below it.


In [51]:
USE_LIVE_FETCH = False   # set to True to pull fresh data from Yahoo Finance

### Option A — Load the included CSV (runs when `USE_LIVE_FETCH = False`)
### Option B — Live fetch from Yahoo Finance (runs when `USE_LIVE_FETCH = True`)

The cell below handles both. Option B uses the `yfinance` package — if you've never installed it, the cell will install it for you (`!pip install yfinance`). If the fetch fails (network down, API rate-limit, etc.), the code automatically falls back to the CSV so the rest of the notebook still works.


In [52]:
import pandas as pd
import numpy as np

def load_data():

    tickers_meta = {
        "^GSPC":  ("S&P 500", "USA"),
        "^DJI":   ("Dow Jones Industrial Average", "USA"),
        "^IXIC":  ("NASDAQ Composite", "USA"),
        "^FTSE":  ("FTSE 100", "United Kingdom"),
        "^N225":  ("Nikkei 225", "Japan"),
        "^GDAXI": ("DAX", "Germany"),
    }

    # -----------------------------
    # 1. Try LIVE DATA
    # -----------------------------
    try:
        import yfinance as yf

        raw = yf.download(
            list(tickers_meta.keys()),
            start="2020-01-02",
            end="2025-01-01",
            auto_adjust=True,
            group_by="ticker",
            progress=False,
        )

        frames = []
        for tk, (name, country) in tickers_meta.items():
            if tk not in raw.columns.get_level_values(0):
                continue

            sub = raw[tk].reset_index()
            sub["Ticker"] = tk
            sub["Name"] = name
            sub["Country"] = country

            frames.append(sub)

        if len(frames) == 0:
            raise ValueError("Live data fetch returned no usable data")

        df = pd.concat(frames, ignore_index=True).dropna(subset=["Close"])

        print(f"Live fetch succeeded — {df.shape[0]:,} rows loaded.")
        return df

    except Exception as e:
        print(f"Live fetch failed: {type(e).__name__}: {e}")

    # -----------------------------
    # 2. SAFE CSV FALLBACK
    # -----------------------------
    try:
        df = pd.read_csv("world_indices.csv")

        # Check if actually valid
        if df.empty or "Date" not in df.columns:
            raise ValueError("CSV is empty or invalid format")

        df["Date"] = pd.to_datetime(df["Date"])

        print(f"Loaded CSV fallback — {df.shape[0]:,} rows.")
        return df

    except Exception as e:
        raise RuntimeError(
            "Both live fetch and CSV fallback failed. "
            "No usable dataset available."
        ) from e


# Load dataset safely
df = load_data()

print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
df.head()



Live fetch succeeded — 7,532 rows loaded.
Date range: 2020-01-02 to 2024-12-31


Price,Date,Open,High,Low,Close,Volume,Ticker,Name,Country
0,2020-01-02,3244.669922,3258.139893,3235.530029,3257.850098,3.459930e+09,^GSPC,S&P 500,USA
1,2020-01-03,3226.360107,3246.149902,3222.340088,3234.850098,3.484700e+09,^GSPC,S&P 500,USA
2,2020-01-06,3217.550049,3246.840088,3214.639893,3246.280029,3.702460e+09,^GSPC,S&P 500,USA
3,2020-01-07,3241.860107,3244.909912,3232.429932,3237.179932,3.435910e+09,^GSPC,S&P 500,USA
4,2020-01-08,3238.590088,3267.070068,3236.669922,3253.050049,3.726840e+09,^GSPC,S&P 500,USA


## Exercise 1 — Initial profile (10 pts)

In the cell below:
1. Print `df.info()`
2. Print the count of rows for each `Ticker`
3. Print the count of missing values per column

Then answer the questions in the next markdown cell.


In [53]:
# TODO: info, rows per ticker, missing per column
# 1. Basic info about dataset
print(df.info())

# 2. Row counts per ticker
print("\nRows per Ticker:")
print(df["Ticker"].value_counts())

# 3. Missing values per column
print("\nMissing values per column:")
print(df.isna().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 7532 entries, 0 to 7810
Data columns (total 9 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   Date     7532 non-null   datetime64[ns]
 1   Open     7532 non-null   float64       
 2   High     7532 non-null   float64       
 3   Low      7532 non-null   float64       
 4   Close    7532 non-null   float64       
 5   Volume   7532 non-null   float64       
 6   Ticker   7532 non-null   object        
 7   Name     7532 non-null   object        
 8   Country  7532 non-null   object        
dtypes: datetime64[ns](1), float64(5), object(3)
memory usage: 588.4+ KB
None

Rows per Ticker:
Ticker
^GDAXI    1275
^FTSE     1261
^DJI      1258
^GSPC     1258
^IXIC     1258
^N225     1222
Name: count, dtype: int64

Missing values per column:
Price
Date       0
Open       0
High       0
Low        0
Close      0
Volume     0
Ticker     0
Name       0
Country    0
dtype: int64


**Answers:**
1. Do all six indices have the same number of rows? Why might they differ?
2. Which column has the most missing values? What's a plausible real-world reason?

YOUR ANSWERS:

1. No, the six indices do not have exactly the same number of rows because each countrys stock market has different trading calendars, holidays, and occasional unavailable data for certain days. This causes slight differences in observations even over the same time period.

2. The column with the most missing values is usually Volume, because index volume is not always consistently reported or tracked across global markets, and in some cases it is estimated, unavailable, or not meaningful for certain indices compared to individual stocks.


## Exercise 2 — Numeric summary by ticker (10 pts)

Produce a table where each row is one ticker, with columns:
- `mean_close`, `std_close`, `min_close`, `max_close`

Round to 2 decimals. Sort by `mean_close` descending.


In [54]:
# TODO: groupby summary by Ticker
summary = df.groupby("Ticker")["Close"].agg(
    mean_close="mean",
    std_close="std",
    min_close="min",
    max_close="max"
)

summary = summary.round(2).sort_values("mean_close", ascending=False)

summary

,mean_close,std_close,min_close,max_close
Ticker,,,,
^DJI,33652.08,4671.36,18591.93,45014.04
^N225,29601.85,5422.84,16552.83,42224.02
^GDAXI,15094.15,2201.69,8441.71,20426.27
^IXIC,13403.60,2662.70,6860.67,20173.89
^FTSE,7265.56,694.76,4993.90,8445.80
^GSPC,4259.61,767.45,2237.40,6090.27


## Exercise 3 — Time series plot (15 pts)

Make a single line plot showing `Close` over `Date`, one line per ticker.

Hint: `sns.lineplot(data=df, x="Date", y="Close", hue="Ticker")` will work, but the indices have very different absolute values (the Nikkei is in the tens of thousands of yen, while the FTSE is in the thousands of pounds), so the lines won't be comparable.

**Fix this** by normalizing each ticker so its first close = 100. This gives a fair comparison of returns. Steps:

1. Sort by Date within each ticker.
2. For each ticker, compute `Close_normalized = Close / first_close * 100`.
3. Plot `Close_normalized` over `Date`, colored by ticker.


In [55]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Ticker", "Date"])

df["Close_normalized"] = df.groupby("Ticker")["Close"].transform(
    lambda x: (x / x.iloc[0]) * 100
)


## Exercise 4 — Daily returns distribution (15 pts)

Compute the **daily percent return** for each ticker: `return = (Close - prev_Close) / prev_Close`.

Hint: within each ticker, use `groupby("Ticker")["Close"].pct_change()`.

Then make a histogram (or KDE plot) showing the distribution of daily returns, with one panel per ticker (a *faceted* plot — `sns.displot` with `col="Ticker"` works well, and `col_wrap=3` to arrange them on two rows).


In [56]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Ticker", "Date"])

df["daily_return"] = df.groupby("Ticker")["Close"].pct_change()


## Exercise 5 — Volatility comparison (15 pts)

The **standard deviation of daily returns** is a common (if simple) measure of an asset's risk / volatility.

1. Compute the standard deviation of daily returns per ticker.
2. Multiply by `√252` to annualize (252 trading days/year).
3. Display as a bar plot, sorted descending.
4. In the markdown cell, identify the most and least volatile index in this dataset.


In [57]:
# TODO: annualized volatility per ticker, bar plot
import numpy as np

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Ticker", "Date"])

df["daily_return"] = df.groupby("Ticker")["Close"].pct_change()

**YOUR ANSWER:** Most volatile = nikkei225, least volatile = Dow Jones Industrial avg.

> `Add blockquote`





## Exercise 6 — Correlation between indices (15 pts)

Do the world's stock markets move together? Compute a correlation matrix of **daily returns** between tickers, and visualize it as a heatmap with `annot=True`.

Hint: you'll need to **pivot** the data first so each ticker is a column. Use `df.pivot(index="Date", columns="Ticker", values="Close").pct_change()`.


In [58]:
# TODO: pivot, compute returns, correlation, heatmap
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df["Date"] = pd.to_datetime(df["Date"])

# reshape so each ticker is a column
wide = df.pivot(index="Date", columns="Ticker", values="Close")

# compute daily returns
returns = wide.pct_change()

/tmp/ipykernel_1172/812392853.py:12: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = wide.pct_change()


**YOUR ANSWER:** Which pair of indices is most correlated? Most weakly correlated? Does the geographic pattern make sense?

The most correlated pair is usually the S&P 500 and NASDAQ because they are all U.S. markets and move together based on the same economic factors. The least correlated pair is usually the Nikkei 225 and a U.S. or European index (like FTSE or NASDAQ) because they are in different regions with different trading hours and economic conditions. Yes, the pattern makes sense: U.S. markets are tightly linked, Europe is also connected, and Japan is the most separate but still positively correlated overall.

## Exercise 7 — Find the worst day (10 pts)

For each ticker, find the date of its **worst daily return** in the dataset. Display ticker, date, and return.

Then in the markdown cell, comment on whether the dates cluster around a common event.


In [59]:
# TODO: worst day per ticker
df["Date"] = pd.to_datetime(df["Date"])

df["daily_return"] = df.groupby("Ticker")["Close"].pct_change()

# index of worst (minimum) return per ticker
worst_idx = df.groupby("Ticker")["daily_return"].idxmin()

worst_days = df.loc[worst_idx, ["Ticker", "Date", "daily_return"]]
worst_days = worst_days.sort_values("daily_return")

worst_days

Price,Ticker,Date,daily_return
1354,^DJI,2020-03-16,-0.129265
6403,^N225,2024-08-05,-0.123958
2656,^IXIC,2020-03-16,-0.123213
6560,^GDAXI,2020-03-12,-0.122386
52,^GSPC,2020-03-16,-0.119841
3956,^FTSE,2020-03-12,-0.108738


**YOUR ANSWER:** Are the worst days clustered? What might explain that?



## Exercise 8 — Reflection (10 pts)

In 4–6 sentences:
1. Compare the EDA workflow on this dataset (time-series, multiple entities) to the one on penguins (cross-sectional, single observation per row). What changed?
2. Name one **modeling question** this exploration would help you set up. (E.g., "Predict whether the S&P will close up tomorrow given recent returns of European indices.")

YOUR ANSWER:

1. compared to the penguins dataset this EDA is more complex because it uses time series data and multiple indices instead of one observation per row. We must sort data by date and compute returns to understand change over time. We also group by ticker to compare different markets instead of comparing individual samples.

2. One modeling question this analysis supports is predicting whether the S&P 500 will go up or down tomorrow using recent returns from other global indices.

## Submission checklist

- [ ] Name and date filled in
- [ ] All TODO cells completed and run
- [ ] All `YOUR ANSWER` prompts replaced
- [ ] **Restart & Run All** completes without errors
- [ ] Downloaded as `.ipynb` and uploaded to Canvas
